# NexaReviews AI — Análise Exploratória de Dados (EDA) e Modelagem

Este notebook apresenta a análise exploratória, higienização textual e validação experimental do pipeline de classificação de sentimentos do projeto **NexaReviews AI**.

In [ ]:
# Configuração do ambiente e importações
import sys
from pathlib import Path

# Adicionar a raiz do projeto ao sys.path para importar módulos de src/
project_root = Path("..").resolve()
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.text_cleaner import TextCleaner
from src.data_processor import DataProcessor
from src.model_pipeline import SentimentModel

# Estilização de gráficos
sns.set_theme(style="whitegrid", palette="muted")
%matplotlib inline
%load_ext watermark
%watermark -v -p pandas,numpy,scikit-learn,seaborn,matplotlib,joblib

## 1. Carregamento dos Dados Brutos
Utilizamos a classe `DataProcessor` para carregar o arquivo CSV a partir de `data/raw/dataset.csv`.

In [ ]:
data_path = project_root / "data" / "raw" / "dataset.csv"
processor = DataProcessor(file_path=str(data_path))
df_raw = processor.load_data()
processor.get_info()

In [ ]:
df_raw.head(10)

## 2. Distribuição das Classes de Sentimento
Visualização da proporção entre avaliações positivas e negativas com `seaborn.countplot`.

In [ ]:
plt.figure(figsize=(7, 4))
palette = {"positivo": "#2ecc71", "negativo": "#e74c3c"}
ax = sns.countplot(data=df_raw, x="sentimento", palette=palette, hue="sentimento", legend=False)
plt.title("Distribuição das Classes de Sentimento", fontsize=14, weight="bold")
plt.xlabel("Sentimento")
plt.ylabel("Contagem")
for p in ax.patches:
    ax.annotate(f"{int(p.get_height())}", (p.get_x() + p.get_width() / 2., p.get_height()),
                ha='center', va='bottom', fontsize=11, weight='bold', xytext=(0, 3), textcoords='offset points')
plt.show()

## 3. Pré-processamento e Limpeza Textual
Aplicação das regras do `TextCleaner` para remoção de acentos, caracteres especiais, pontuações e normalização minúscula.

In [ ]:
df_clean = processor.clean_data(
    text_column="texto_review",
    target_column="sentimento",
    save_path=str(project_root / "data" / "processed" / "dataset_cleaned.csv")
)
df_clean.head()

### Comparativo: Texto Original vs. Texto Limpo

In [ ]:
for i in range(min(5, len(df_clean))):
    print(f"[ORIGINAL] {df_clean.loc[i, 'texto_review']}")
    print(f"[LIMPO]    {df_clean.loc[i, 'texto_limpo']}")
    print("-" * 80)

## 4. Análise de Comprimento dos Textos
Análise de distribuição do número de caracteres e palavras por sentimento.

In [ ]:
df_clean["num_palavras"] = df_clean["texto_limpo"].apply(lambda x: len(x.split()))
df_clean["num_caracteres"] = df_clean["texto_limpo"].apply(len)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.boxplot(data=df_clean, x="sentimento", y="num_palavras", palette=palette, ax=axes[0])
axes[0].set_title("Distribuição do Número de Palavras por Sentimento")

sns.kdeplot(data=df_clean, x="num_caracteres", hue="sentimento", palette=palette, fill=True, ax=axes[1])
axes[1].set_title("Densidade do Comprimento de Caracteres")
plt.tight_layout()
plt.show()

## 5. Treinamento e Avaliação do Modelo
Instanciamos `SentimentModel` com pipeline Scikit-Learn (`TfidfVectorizer` + `LogisticRegression`).

In [ ]:
model = SentimentModel(max_features=5000, random_state=42)
metrics = model.train(X=df_clean["texto_limpo"], y=df_clean["sentimento"], test_size=0.2, random_state=42)

# Salvar o modelo treinado
model.save_model(str(project_root / "models" / "sentiment_model.pkl"))

### Visualização da Matriz de Confusão

In [ ]:
cm = metrics["confusion_matrix"]
classes = np.unique(df_clean["sentimento"])

plt.figure(figsize=(6, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=classes, yticklabels=classes)
plt.title("Matriz de Confusão - NexaReviews AI", fontsize=13, weight="bold")
plt.xlabel("Predição do Modelo")
plt.ylabel("Valor Real")
plt.show()

## 6. Teste de Inferência em Tempo Real

In [ ]:
exemplos = [
    "Adorei o produto, funcionou perfeitamente e chegou super rápido!",
    "Muito ruim, quebrou na primeira semana e a loja não deu garantia.",
    "Bom custo-benefício, recomendo."
]

for ex in exemplos:
    ex_limpo = TextCleaner.clean_text(ex)
    pred = model.predict(ex_limpo)[0]
    prob = model.predict_proba(ex_limpo)[0]
    print(f"Original: {ex}")
    print(f"Predição: >>> {pred.upper()} <<< | Probabilidades: {dict(zip(model.pipeline.classes_, prob))}")
    print("-" * 70)